# 08. Single-Notebook OpenClaw LangGraph Project

Этот notebook собирает весь демонстрационный OpenClaw-проект в одном месте: настройки, prompts, connectors, LangGraph assistants и команды запуска.

Ограничение честное: `langgraph dev` запускает Python entrypoint, а не `.ipynb` напрямую. Поэтому notebook является source of truth, а ячейка ниже генерирует маленький runtime-файл и `langgraph.notebook.json`. Запуск все равно идет через LangGraph.


## Presentation Visuals

These frames come from the local presentation assets and make the notebook usable as a walkthrough, not only as executable code.

![OpenClaw workshop thumbnail](../presentation/rendered/openclawtype_workshop_thumb.png)

![Russian Manim presentation thumbnail](../openclawtype_ru_manim/thumbnail.png)


## Что находится внутри

- demo issue connector;
- Telegram connector;
- Jenkins connector;
- VK connector;
- base agent и SWE agent;
- filesystem backend с opt-in shell mode;
- env template без секретов;
- `langgraph dev` config.


In [ ]:
from pathlib import Path
import json

REPO_ROOT = Path.cwd()
RUNTIME_DIR = REPO_ROOT / ".openclaw_notebook_runtime"
RUNTIME_DIR.mkdir(exist_ok=True)

agent_py = '"""Generated from workshop_notebooks/08_single_notebook_langgraph_project.ipynb.\n\nDo not edit this file directly. Re-run the notebook cell that writes the\nruntime if you want to change the generated LangGraph entrypoint.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport random\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nfrom typing import Any\nfrom urllib.parse import parse_qsl, urlencode, urljoin, urlparse, urlunparse\n\nimport httpx\nfrom deepagents import create_deep_agent\nfrom dotenv import load_dotenv\nfrom langgraph_sdk import get_sync_client\nfrom langchain_core.tools import tool\n\nload_dotenv()\n\nDEFAULT_MODEL = "openrouter:tencent/hy3:free"\nDEFAULT_MINIMAX_BASE_URL = "https://api.minimaxi.chat/v1"\nDEFAULT_MINIMAX_MODEL = "MiniMax-M2"\nDEFAULT_DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"\nDEFAULT_DEEPSEEK_MODEL = "deepseek-v4-flash"\nDEFAULT_JENKINS_JOB_URL = "https://devops.brojs.ru/job/marat/"\nJENKINS_TIMEOUT = 20.0\nTELEGRAM_API_BASE = "https://api.telegram.org"\nTELEGRAM_TIMEOUT = 15.0\nVK_API_BASE = "https://api.vk.com/method"\nDEFAULT_VK_API_VERSION = "5.199"\nVK_TIMEOUT = 20.0\nDEFAULT_LANGGRAPH_URL = "http://127.0.0.1:2024"\nDEFAULT_ASSISTANT_ID = "openclaw_notebook"\n\n\ndef _env(name: str) -> str | None:\n    value = os.getenv(name, "").strip()\n    return value or None\n\n\ndef _json(data: dict[str, Any] | list[Any]) -> str:\n    return json.dumps(data, ensure_ascii=False, indent=2)\n\n\ndef _mask_secret(secret: str | None) -> str:\n    if not secret:\n        return "<missing>"\n    if len(secret) <= 8:\n        return "<set>"\n    return f"{secret[:4]}...{secret[-4:]}"\n\n\ndef _model():\n    provider = os.getenv("OPENCLAW_PROVIDER")\n    if provider == "minimax":\n        from langchain_openai import ChatOpenAI\n\n        return ChatOpenAI(\n            model=os.getenv("MINIMAX_MODEL", DEFAULT_MINIMAX_MODEL),\n            api_key=os.getenv("MINIMAX_API_KEY"),\n            base_url=os.getenv("MINIMAX_BASE_URL", DEFAULT_MINIMAX_BASE_URL),\n            temperature=0,\n        )\n\n    if provider == "deepseek":\n        from langchain_openai import ChatOpenAI\n\n        return ChatOpenAI(\n            model=os.getenv("DEEPSEEK_MODEL", DEFAULT_DEEPSEEK_MODEL),\n            api_key=os.getenv("DEEPSEEK_API_KEY"),\n            base_url=os.getenv("DEEPSEEK_BASE_URL", DEFAULT_DEEPSEEK_BASE_URL),\n            temperature=0,\n        )\n\n    return os.getenv("OPENCLAW_MODEL", DEFAULT_MODEL)\n\n\ndef _workspace_root() -> Path:\n    return Path(os.getenv("OPENCLAW_WORKSPACE", ".")).expanduser().resolve()\n\n\ndef _backend():\n    root_dir = _workspace_root()\n    if os.getenv("OPENCLAW_ENABLE_LOCAL_SHELL") != "1":\n        from deepagents.backends import FilesystemBackend\n\n        return FilesystemBackend(root_dir=root_dir, virtual_mode=True)\n\n    from deepagents.backends import LocalShellBackend\n\n    return LocalShellBackend(\n        root_dir=root_dir,\n        virtual_mode=True,\n        inherit_env=True,\n        timeout=120,\n        max_output_bytes=80_000,\n    )\n\n\n@dataclass(frozen=True)\nclass DemoIssue:\n    id: str\n    title: str\n    status: str\n    priority: str\n    owner: str\n    summary: str\n\n\nDEMO_ISSUES = [\n    DemoIssue(\n        id="DEMO-101",\n        title="Add connector onboarding flow",\n        status="ready",\n        priority="high",\n        owner="workshop",\n        summary="Show how a connector becomes an agent tool without changing the Deep Agents runtime.",\n    ),\n    DemoIssue(\n        id="DEMO-102",\n        title="Enable local shell mode guardrails",\n        status="backlog",\n        priority="medium",\n        owner="platform",\n        summary="Document the risks and add an approval checklist before enabling shell mode.",\n    ),\n]\n\n\n@tool\ndef list_demo_issues() -> str:\n    """List issues from the demo issue tracker connector."""\n    return _json([asdict(issue) for issue in DEMO_ISSUES])\n\n\n@tool\ndef get_demo_issue(issue_id: str) -> str:\n    """Fetch one issue from the demo issue tracker connector by id, for example DEMO-101."""\n    normalized_id = issue_id.strip().upper()\n    for issue in DEMO_ISSUES:\n        if issue.id == normalized_id:\n            return _json(asdict(issue))\n    known_ids = ", ".join(issue.id for issue in DEMO_ISSUES)\n    return f"Unknown issue id {issue_id!r}. Known ids: {known_ids}"\n\n\ndef _telegram_token() -> str | None:\n    return _env("TELEGRAM_BOT_TOKEN")\n\n\ndef _default_chat_id() -> str | None:\n    return _env("TELEGRAM_CHAT_ID")\n\n\ndef _telegram_url(method: str, token: str) -> str:\n    return f"{TELEGRAM_API_BASE}/bot{token}/{method}"\n\n\n@tool\ndef send_telegram_message(message: str, chat_id: str | None = None, dry_run: bool = True) -> str:\n    """Send a message through Telegram Bot API, or preview it when dry_run is true."""\n    resolved_chat_id = (chat_id or _default_chat_id() or "").strip()\n    payload = {\n        "chat_id": resolved_chat_id or "<missing TELEGRAM_CHAT_ID>",\n        "text": message,\n        "disable_web_page_preview": True,\n    }\n\n    if dry_run:\n        return _json({"dry_run": True, "method": "sendMessage", "payload": payload})\n\n    token = _telegram_token()\n    if not token:\n        return "TELEGRAM_BOT_TOKEN is not set. Set it in .env or call with dry_run=True."\n    if not resolved_chat_id:\n        return "Telegram chat_id is missing. Pass chat_id or set TELEGRAM_CHAT_ID in .env."\n\n    response = httpx.post(_telegram_url("sendMessage", token), json=payload, timeout=TELEGRAM_TIMEOUT)\n    if response.is_error:\n        return _json({"ok": False, "status_code": response.status_code, "response": response.text})\n    return _json(response.json())\n\n\n@tool\ndef get_telegram_updates(limit: int = 5) -> str:\n    """Read recent Telegram Bot API updates for the configured bot token."""\n    token = _telegram_token()\n    if not token:\n        return "TELEGRAM_BOT_TOKEN is not set. Add it to .env to read updates."\n\n    bounded_limit = max(1, min(limit, 20))\n    response = httpx.get(_telegram_url("getUpdates", token), params={"limit": bounded_limit}, timeout=TELEGRAM_TIMEOUT)\n    if response.is_error:\n        return _json({"ok": False, "status_code": response.status_code, "response": response.text})\n    return _json(response.json())\n\n\ndef _job_url(job_url: str | None = None) -> str:\n    return (job_url or _env("JENKINS_JOB_URL") or DEFAULT_JENKINS_JOB_URL).rstrip("/") + "/"\n\n\ndef _job_token() -> str | None:\n    return _env("JENKINS_JOB_TOKEN")\n\n\ndef _auth() -> tuple[str, str] | None:\n    username = _env("JENKINS_USERNAME")\n    api_token = _env("JENKINS_API_TOKEN")\n    if username and api_token:\n        return username, api_token\n    return None\n\n\ndef _with_query(url: str, params: dict[str, str]) -> str:\n    parsed = urlparse(url)\n    query = dict(parse_qsl(parsed.query, keep_blank_values=True))\n    query.update(params)\n    return urlunparse(parsed._replace(query=urlencode(query)))\n\n\ndef _jenkins_root_url(job_url: str) -> str:\n    parsed = urlparse(job_url)\n    return urlunparse((parsed.scheme, parsed.netloc, "/", "", "", ""))\n\n\ndef _crumb_headers(client: httpx.Client, base_url: str) -> dict[str, str]:\n    crumb_url = urljoin(_jenkins_root_url(base_url), "crumbIssuer/api/json")\n    response = client.get(crumb_url)\n    if response.status_code == 404:\n        return {}\n    response.raise_for_status()\n    payload = response.json()\n    field = payload.get("crumbRequestField")\n    crumb = payload.get("crumb")\n    if isinstance(field, str) and isinstance(crumb, str):\n        return {field: crumb}\n    return {}\n\n\n@tool\ndef get_jenkins_job_info(job_url: str | None = None) -> str:\n    """Read basic metadata for the configured Jenkins job."""\n    resolved_job_url = _job_url(job_url)\n    api_url = urljoin(resolved_job_url, "api/json")\n    params = {"tree": "name,url,buildable,color,lastBuild[number,url,result,timestamp],lastSuccessfulBuild[number,url]"}\n\n    try:\n        response = httpx.get(api_url, params=params, auth=_auth(), timeout=JENKINS_TIMEOUT)\n        response.raise_for_status()\n    except httpx.HTTPError as exc:\n        return _json({"ok": False, "job_url": resolved_job_url, "error": str(exc)})\n\n    return _json({"ok": True, "job_url": resolved_job_url, "job": response.json()})\n\n\n@tool\ndef trigger_jenkins_job(\n    job_url: str | None = None,\n    parameters: dict[str, str] | None = None,\n    dry_run: bool = False,\n) -> str:\n    """Trigger the configured Jenkins job, or preview the request when dry_run is true."""\n    resolved_job_url = _job_url(job_url)\n    build_path = "buildWithParameters" if parameters else "build"\n    build_url = urljoin(resolved_job_url, build_path)\n    token = _job_token()\n\n    query_params = {key: str(value) for key, value in (parameters or {}).items()}\n    if token:\n        query_params["token"] = token\n    request_url = _with_query(build_url, query_params) if query_params else build_url\n\n    preview = {\n        "job_url": resolved_job_url,\n        "method": "POST",\n        "endpoint": build_path,\n        "uses_job_token": bool(token),\n        "job_token": _mask_secret(token),\n        "uses_basic_auth": bool(_auth()),\n        "parameters": parameters or {},\n    }\n\n    if dry_run:\n        return _json({"dry_run": True, **preview})\n\n    if not token and not _auth():\n        return "Jenkins credentials are not set. Set JENKINS_JOB_TOKEN or JENKINS_USERNAME/JENKINS_API_TOKEN, or call with dry_run=True."\n\n    try:\n        with httpx.Client(auth=_auth(), timeout=JENKINS_TIMEOUT) as client:\n            headers = _crumb_headers(client, resolved_job_url) if _auth() else {}\n            response = client.post(request_url, headers=headers)\n            response.raise_for_status()\n    except httpx.HTTPError as exc:\n        return _json({"ok": False, **preview, "error": str(exc)})\n\n    return _json({"ok": True, **preview, "status_code": response.status_code, "queue_url": response.headers.get("Location")})\n\n\ndef _vk_access_token() -> str | None:\n    return _env("VK_ACCESS_TOKEN")\n\n\ndef _vk_api_version() -> str:\n    return _env("VK_API_VERSION") or DEFAULT_VK_API_VERSION\n\n\ndef _vk_method_url(method: str) -> str:\n    normalized = method.strip().lstrip("/")\n    return f"{VK_API_BASE}/{normalized}"\n\n\ndef _vk_request_payload(params: dict[str, Any] | None = None) -> dict[str, Any]:\n    payload = {key: value for key, value in (params or {}).items() if value is not None}\n    payload["v"] = _vk_api_version()\n    token = _vk_access_token()\n    if token:\n        payload["access_token"] = token\n    return payload\n\n\ndef _vk_preview_payload(payload: dict[str, Any]) -> dict[str, Any]:\n    preview = dict(payload)\n    preview["access_token"] = _mask_secret(_vk_access_token())\n    return preview\n\n\n@tool\ndef call_vk_api_method(method: str, params: dict[str, Any] | None = None, dry_run: bool = True) -> str:\n    """Call a VK API method, or preview the request when dry_run is true."""\n    payload = _vk_request_payload(params)\n    preview = {\n        "method": method.strip(),\n        "url": _vk_method_url(method),\n        "uses_access_token": bool(_vk_access_token()),\n        "payload": _vk_preview_payload(payload),\n    }\n\n    if dry_run:\n        return _json({"dry_run": True, **preview})\n\n    if not _vk_access_token():\n        return "VK_ACCESS_TOKEN is not set. Add it to .env or call with dry_run=True."\n\n    try:\n        response = httpx.post(_vk_method_url(method), data=payload, timeout=VK_TIMEOUT)\n        response.raise_for_status()\n    except httpx.HTTPError as exc:\n        return _json({"ok": False, **preview, "error": str(exc)})\n\n    data = response.json()\n    if "error" in data:\n        return _json({"ok": False, **preview, "response": data})\n    return _json({"ok": True, **preview, "response": data})\n\n\n@tool\ndef get_vk_current_user(dry_run: bool = False) -> str:\n    """Read the VK profile visible to the configured access token."""\n    return call_vk_api_method.invoke({"method": "users.get", "params": {"fields": "screen_name,photo_100"}, "dry_run": dry_run})\n\n\n@tool\ndef send_vk_message(peer_id: str, message: str, random_id: int | None = None, dry_run: bool = True) -> str:\n    """Send a VK message, or preview it when dry_run is true."""\n    resolved_random_id = random_id if random_id is not None else random.randint(1, 2_147_483_647)\n    return call_vk_api_method.invoke(\n        {\n            "method": "messages.send",\n            "params": {"peer_id": peer_id, "message": message, "random_id": resolved_random_id},\n            "dry_run": dry_run,\n        }\n    )\n\n\ndef _langgraph_url(url: str | None = None) -> str:\n    return url or _env("LANGGRAPH_URL") or DEFAULT_LANGGRAPH_URL\n\n\ndef _assistant_id(assistant: str | None = None) -> str:\n    return assistant or _env("LANGGRAPH_ASSISTANT_ID") or DEFAULT_ASSISTANT_ID\n\n\ndef _build_agent_input(\n    message: str,\n    *,\n    source: str,\n    metadata: dict[str, Any] | None = None,\n) -> dict:\n    context = {\n        "source": source,\n        **(metadata or {}),\n    }\n    return {\n        "messages": [\n            {\n                "role": "user",\n                "content": f"External message context: {json.dumps(context, ensure_ascii=False)}\\n\\n{message}",\n            }\n        ]\n    }\n\n\n@tool\ndef trigger_langgraph_from_vk_message(\n    peer_id: str,\n    message: str,\n    vk_message_id: str | None = None,\n    assistant_id: str | None = None,\n    langgraph_url: str | None = None,\n    wait: bool = True,\n    dry_run: bool = True,\n) -> str:\n    """Trigger a LangGraph assistant run from a VK message."""\n    resolved_url = _langgraph_url(langgraph_url)\n    resolved_assistant = _assistant_id(assistant_id)\n    agent_input = _build_agent_input(\n        message,\n        source="vk",\n        metadata={\n            "peer_id": peer_id,\n            "vk_message_id": vk_message_id,\n        },\n    )\n    preview = {\n        "langgraph_url": resolved_url,\n        "assistant_id": resolved_assistant,\n        "input": agent_input,\n        "wait": wait,\n    }\n\n    if dry_run:\n        return _json({"dry_run": True, **preview})\n\n    client = get_sync_client(url=resolved_url)\n    thread = client.threads.create(\n        metadata={\n            "source": "vk",\n            "peer_id": peer_id,\n            "vk_message_id": vk_message_id,\n        }\n    )\n    run = client.runs.create(\n        thread["thread_id"],\n        resolved_assistant,\n        input=agent_input,\n        stream_mode="values",\n        multitask_strategy="enqueue",\n    )\n    result: dict[str, Any] = {\n        "ok": True,\n        **preview,\n        "thread_id": thread["thread_id"],\n        "run_id": run["run_id"],\n    }\n    if wait:\n        result["output"] = client.runs.wait(thread["thread_id"], run["run_id"])\n    return _json(result)\n\n\nCONNECTOR_TOOLS = [\n    list_demo_issues,\n    get_demo_issue,\n    send_telegram_message,\n    get_telegram_updates,\n    get_jenkins_job_info,\n    trigger_jenkins_job,\n    call_vk_api_method,\n    get_vk_current_user,\n    send_vk_message,\n    trigger_langgraph_from_vk_message,\n]\n\nBASE_PROMPT = """\\\nYou are OpenClaw, an experimental open-source coding agent built with LangChain\nDeep Agents. You help with software engineering, repository navigation, product\nresearch, implementation, and DevOps checks.\n\nWork like a careful senior engineer:\n- inspect before editing;\n- keep changes focused;\n- verify with tests or equivalent checks;\n- explain only the useful result to the user.\n\nConnector rules:\n- use dry-run mode before mutating Telegram, Jenkins, or VK unless the user explicitly asks for a real action;\n- never print full secrets;\n- if Jenkins network access fails, report it as a network/VPN/firewall issue after a dry-run succeeds.\n"""\n\nSWE_PROMPT = BASE_PROMPT + """\\\n\nYou are running in SWE mode. Optimize for issue resolution:\n- reproduce or characterize the failure first;\n- localize the root cause before patching;\n- add regression coverage where practical;\n- run the narrowest useful verification before broad checks;\n- keep a clear chain from issue to patch to test.\n"""\n\nSUBAGENTS = [\n    {\n        "name": "repo-researcher",\n        "description": "Map repository structure, APIs, tests, and likely change locations.",\n        "system_prompt": "You research codebases. Inspect files, identify relevant modules, and return concise findings with file paths and rationale.",\n    },\n    {\n        "name": "reviewer",\n        "description": "Review proposed patches for bugs, missing tests, and regressions.",\n        "system_prompt": "You are a code reviewer. Focus on correctness, edge cases, tests, security, and maintainability. Be specific and concise.",\n    },\n]\n\nagent = create_deep_agent(\n    model=_model(),\n    tools=CONNECTOR_TOOLS,\n    system_prompt=BASE_PROMPT,\n    subagents=SUBAGENTS,\n    backend=_backend(),\n    interrupt_on={"execute": True},\n)\n\nswe_agent = create_deep_agent(\n    model=_model(),\n    tools=CONNECTOR_TOOLS,\n    system_prompt=SWE_PROMPT,\n    subagents=SUBAGENTS,\n    backend=_backend(),\n    interrupt_on={"execute": True, "write_file": True, "edit_file": True},\n)\n'
(RUNTIME_DIR / "agent.py").write_text(agent_py)

langgraph_config = {
    "dependencies": ["."],
    "graphs": {
        "openclaw_notebook": "./.openclaw_notebook_runtime/agent.py:agent",
        "openclaw_notebook_swe": "./.openclaw_notebook_runtime/agent.py:swe_agent",
    },
    "env": ".env",
}
Path("langgraph.notebook.json").write_text(json.dumps(langgraph_config, indent=2) + "\n")

notebook_env_example = """# Copy relevant values into .env. Do not commit real secrets.
OPENCLAW_MODEL=openrouter:tencent/hy3:free
OPENROUTER_API_KEY=
OPENAI_API_KEY=
ANTHROPIC_API_KEY=
LANGSMITH_TRACING=false
OPENCLAW_ENABLE_LOCAL_SHELL=0
OPENCLAW_WORKSPACE=.
TELEGRAM_BOT_TOKEN=
TELEGRAM_CHAT_ID=
JENKINS_JOB_URL=https://devops.brojs.ru/job/marat/
JENKINS_JOB_TOKEN=
JENKINS_USERNAME=
JENKINS_API_TOKEN=
OPENCLAW_RUN_REAL_JENKINS_PIPELINE=0
VK_ACCESS_TOKEN=
VK_API_VERSION=5.199
LANGGRAPH_URL=http://127.0.0.1:2024
LANGGRAPH_ASSISTANT_ID=openclaw_notebook
VK_BRIDGE_DRY_RUN=1
VK_BRIDGE_REPLY_TO_VK=0
VK_BRIDGE_WAIT_FOR_RUN=1
VK_BRIDGE_POLL_SECONDS=5
VK_BRIDGE_CONVERSATION_COUNT=10
VK_BRIDGE_STATE_PATH=.openclaw_vk_bridge_state.json
"""
(RUNTIME_DIR / ".env.example").write_text(notebook_env_example)

print(f"Wrote {RUNTIME_DIR / 'agent.py'}")
print("Wrote langgraph.notebook.json")
print("Run: uv run langgraph dev --config langgraph.notebook.json")


## Validate generated runtime

Эта ячейка проверяет, что сгенерированный LangGraph entrypoint импортируется и оба assistants собираются.


In [ ]:
import importlib.util
import sys
from pathlib import Path

runtime_path = Path('.openclaw_notebook_runtime/agent.py')
spec = importlib.util.spec_from_file_location('openclaw_notebook_runtime_agent', runtime_path)
module = importlib.util.module_from_spec(spec)
assert spec.loader is not None
sys.modules[spec.name] = module
spec.loader.exec_module(module)

print(type(module.agent).__name__)
print(type(module.swe_agent).__name__)
print([tool.name for tool in module.CONNECTOR_TOOLS])


## Safe connector smoke checks

Эта ячейка не делает внешних мутаций: demo connector локальный, Jenkins и VK работают в dry-run.


In [ ]:
print(module.list_demo_issues.invoke({}))
print(module.trigger_jenkins_job.invoke({'parameters': {'OPENCLAW_SMOKE': 'true'}, 'dry_run': True}))
print(module.send_vk_message.invoke({'peer_id': '123', 'message': 'OpenClaw notebook dry-run', 'random_id': 42, 'dry_run': True}))


## Run through LangGraph

В терминале из корня проекта:

```bash
uv run langgraph dev --config langgraph.notebook.json
```

В Deep Agents UI используй:

- Deployment URL: `http://127.0.0.1:2024`
- Assistant ID: `openclaw_notebook` или `openclaw_notebook_swe`


## Demo prompts

```text
Use the demo issue connector. List issues, fetch DEMO-101, and propose the next implementation step.
```

```text
Use the Jenkins connector in dry-run mode with parameter OPENCLAW_SMOKE=true. Show the request that would be sent.
```

```text
Use the VK connector. First prepare a dry-run message payload for peer_id PEER_ID that says "OpenClaw VK connector demo". Then wait for my confirmation before sending it with dry_run=false.
```


## VK -> LangGraph Bridge

The VK connector can now trigger the running LangGraph assistant through `langgraph-sdk`. Start LangGraph first:

```bash
uv run langgraph dev --config langgraph.notebook.json
```

Preview a VK-originated trigger without starting a run:

```python
module.trigger_langgraph_from_vk_message.invoke({
    "peer_id": "123",
    "message": "Проверь статус проекта",
    "vk_message_id": "demo-1",
    "dry_run": True,
})
```

Run the polling bridge once in dry-run mode:

```bash
VK_BRIDGE_ONCE=1 VK_BRIDGE_DRY_RUN=1 uv run python scripts/vk_langgraph_bridge.py
```

For real VK -> LangGraph -> VK replies, set `VK_BRIDGE_DRY_RUN=0` and `VK_BRIDGE_REPLY_TO_VK=1` after validating the payload.
